# NIH Digital Mental Health RAG Proposal Generator
## PAR-25-136 · Laboratories to Optimize Digital Health (R01)

---

## **Pipeline Overview**

1. Configuration & Setup
2. API & Embedding Helpers
3. Parse the NOFO
4. Parse Research Papers
5. Build FAISS Vector Index
6. Filter Papers by Topical Relevance to NOFO
7. Generate Bridging Research Ideas
8. Generate NIH Technical Approach
9. LLM Self-Evaluation Loop
10. Human Evaluation Checkpoint
11. Human-Guided Revision
12. Final Proposal Output
13. Pipeline Summary & Audit Trail
14. PDF Generation

---

## **Configuration & Setup**

This cell loads all required libraries, reads the OpenAI API key and base URL from `config.json`, defines file paths for the NOFO and research papers, and sets all tunable pipeline parameters. It also sets `RUN_MODE` to either `"auto"` (full pipeline runs without human intervention, used for baseline comparison) or `"human"` (pipeline pauses at the review checkpoint for editorial feedback, used for the final submission draft).

In [15]:
import os
import json
import re
import textwrap
import time
import numpy as np
import faiss
import pdfplumber
import requests
from pathlib import Path
from typing import List, Dict, Tuple

# ── Load API key from config.json ─────────────────────────────────────────────
CONFIG_PATH = Path(__file__).parent / "config.json" if "__file__" in dir() else Path("config.json")
with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

OPENAI_API_KEY  = config["API_KEY"]
OPENAI_API_BASE = config["OPENAI_API_BASE"]
OPENAI_MODEL    = "gpt-4o"  # or gpt-4-turbo, gpt-3.5-turbo, etc.

# ── File paths ────────────────────────────────────────────────────────────────
BASE_PATH   = Path(".")         # current directory — same folder as the notebook
PAPERS_PATH = BASE_PATH / "Papers"  # subfolder containing all research PDFs
NOFO_PATH   = BASE_PATH / "PAR 25 136 Laboratories to Optimize Digital Health.pdf"
PAPER_PATHS = list(PAPERS_PATH.glob("*.pdf"))  # automatically picks up all PDFs in the folder

# ── Chunking parameters ───────────────────────────────────────────────────────
CHUNK_SIZE    = 400   # tokens ≈ words (rough)
CHUNK_OVERLAP = 75

# ── Retrieval / generation parameters ────────────────────────────────────────
TOP_K_NOFO          = 4     # chunks from NOFO to include in context
TOP_K_PAPERS        = 3     # top paper chunks per query
RELEVANCE_THRESHOLD = 0.45  # cosine similarity floor for paper filtering
MAX_EVAL_LOOPS      = 3     # max LLM self-evaluation passes

# ── Run mode ──────────────────────────────────────────────────────────────────
# "auto"  : full pipeline runs without any human approval checkpoint
#           useful for baseline comparison and initial review
# "human" : pipeline pauses at human checkpoint for review and feedback
#           use this for the final submission draft
RUN_MODE = "human"  # switch to "auto" for baseline run

print("Configuration loaded.")
print(f"  Config file : {CONFIG_PATH}")
print(f"  API key     : {OPENAI_API_KEY[:8]}{'*' * 20}  (truncated for display)")
print(f"  Run mode    : {RUN_MODE.upper()}")
print(f"  NOFO        : {NOFO_PATH.name}")
print(f"  Papers      : {[p.name for p in PAPER_PATHS]}")

Configuration loaded.
  Config file : config.json
  API key     : sk-proj-********************  (truncated for display)
  Run mode    : HUMAN
  NOFO        : PAR 25 136 Laboratories to Optimize Digital Health.pdf
  Papers      : ['2021_EPJ_MVMCInfoOps.pdf', '2024_ICWSM_Data_Challenge__Post_API_Data_Collection.pdf', '23-US-DHS-001.pdf', 'AAAI IAA CV.pdf', 'Acquiring Maintainable AI_Enable Systems_Final.pdf', 'ALL18.pdf', 'Analysis_of_Malware_Communities_Using_Multi_Modal_Features.pdf', 'Arrow White Paper DExTra.pdf', 'A_Complex_Network_Approach_to_Find_Latent_Terorrist_Communities.pdf', 'Benson_MA491_NLP.pdf', 'BotBuster___AAAI.pdf', 'CausalOrgInorgContent.pdf', 'Characterizing_Communities_of_Hashtag_Usage_on_Twitter_During_the_2020_COVID_19_Pandemic.pdf', 'Chat GPT Bias final w copyright.pdf', 'ClassifiersCrowdSource.pdf', 'Clustering_Analysis_of_Website_Usage_on_Twitter_during_the_COVID_19_Pandemic.pdf', 'Cohort_Optimization_Methods_SNAMS_2021_working_draft (4).pdf', 'Confidence_Chain

---

## **API & Embedding Helpers**

This cell defines three core helper functions used throughout the pipeline. `llm_complete()` handles all OpenAI API calls with automatic exponential backoff on rate limit errors (429s). `get_embedding()` produces a 1536-dimensional hash-projection vector for a given text string — a lightweight, dependency-free approach that requires no external model; for higher retrieval quality this can be swapped for OpenAI or sentence-transformers embeddings. `embed_batch()` applies `get_embedding()` across a list of text chunks and returns a stacked numpy array suitable for FAISS indexing.

In [16]:
def llm_complete(messages: List[Dict], system: str = "", max_tokens: int = 2048,
                 max_retries: int = 5, base_wait: float = 10.0) -> str:
    """Call the OpenAI Chat Completions API with exponential backoff on 429s."""
    headers = {
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json",
    }
    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    body = {
        "model": OPENAI_MODEL,
        "max_tokens": max_tokens,
        "messages": full_messages,
    }

    for attempt in range(1, max_retries + 1):
        resp = requests.post(f"{OPENAI_API_BASE}/chat/completions",
                             headers=headers, json=body, timeout=120)
        if resp.status_code == 429:
            wait = base_wait * (2 ** (attempt - 1))  # 10s, 20s, 40s, 80s, 160s
            print(f"  ⚠ Rate limited (429). Waiting {wait:.0f}s before retry "
                  f"{attempt}/{max_retries} ...")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp.json()["choices"][0]["message"]["content"]

    raise RuntimeError(f"Failed after {max_retries} retries due to rate limiting.")

def get_embedding(text: str) -> np.ndarray:
    """
    Lightweight hash-projection embedding producing a 1536-dim float32 vector
    via character n-gram and word unigram hashing — no external model needed.

    For production use, swap this with OpenAI text-embedding-3-small or a
    local sentence-transformers model (e.g., all-MiniLM-L6-v2).
    """
    DIM = 1536
    vec = np.zeros(DIM, dtype=np.float32)
    text = text.lower()
    # character 3-grams + word unigrams
    tokens = text.split()
    ngrams = [text[i:i+3] for i in range(len(text)-2)]
    for tok in tokens + ngrams:
        h = hash(tok) % DIM
        vec[h] += 1.0
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec /= norm
    return vec

def embed_batch(texts: List[str]) -> np.ndarray:
    """Embed a list of text chunks, returning a stacked numpy array for FAISS."""
    if not texts:  # guard against empty chunk list
        return np.zeros((0, DIM), dtype=np.float32)
    return np.vstack([get_embedding(t) for t in texts])

print("Helper functions defined.")
print("NOTE: Using hash-projection embeddings (no external model required).")
print("      For higher retrieval quality, replace get_embedding() with a")
print("      sentence-transformers or OpenAI embeddings call.")

Helper functions defined.
NOTE: Using hash-projection embeddings (no external model required).
      For higher retrieval quality, replace get_embedding() with a
      sentence-transformers or OpenAI embeddings call.


---

## **Parse the NOFO**

This cell defines two reusable text processing functions used throughout the pipeline. `extract_pdf_text()` reads a PDF file page by page using `pdfplumber` and returns the full extracted text as a single string. `chunk_text()` splits that text into overlapping word-count chunks of size `CHUNK_SIZE` with `CHUNK_OVERLAP` words of overlap between adjacent chunks, ensuring that sentences spanning chunk boundaries are not lost during retrieval. The NOFO PDF is then parsed, chunked, and previewed to confirm successful extraction.

In [17]:
def extract_pdf_text(path: Path) -> str:
    """Extract full text from a PDF using pdfplumber."""
    pages = []
    with pdfplumber.open(str(path)) as pdf:
        for page in pdf.pages:
            txt = page.extract_text()
            if txt:
                pages.append(txt)
    return "\n\n".join(pages)

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE,
               overlap: int = CHUNK_OVERLAP) -> List[str]:
    """Split text into overlapping word-count chunks."""
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

# ── Parse NOFO ────────────────────────────────────────────────────────────────
print(f"Parsing NOFO: {NOFO_PATH.name} ...")
nofo_text   = extract_pdf_text(NOFO_PATH)
nofo_chunks = chunk_text(nofo_text)

print(f"  Total characters : {len(nofo_text):,}")
print(f"  Total chunks     : {len(nofo_chunks)}")
print()
print("=== NOFO Preview (first 800 chars) ===")
print(nofo_text[:800])

Parsing NOFO: PAR 25 136 Laboratories to Optimize Digital Health.pdf ...
  Total characters : 80,573
  Total chunks     : 32

=== NOFO Preview (first 800 chars) ===
Department of Health and Human Services
Part 1. Overview Information
Participating Organization(s) National Institutes of Health (NIH (http://www.nih.gov))
Components of Participating Organizations National Institute of Mental Health (NIMH (https://www.nimh.nih.gov/index.shtml))
Funding Opportunity Title Laboratories to Optimize Digital Health (R01 Clinical Trial Required)
Activity Code R01 (//grants.nih.gov/grants/funding/ac_search_results.htm?
text_curr=r01&Search.x=0&Search.y=0&Search_Type=Activity) Research Project Grant
Announcement Type Reissue of PAR-23-096 (https://grants.nih.gov/grants/guide/pa-files/PAR-23-096.html)
Related Notices See Notices of Special Interest (https://grants.nih.gov/grants/guide/NOSIs_targetingList.cfm?GuideDocID=41794)
associated with this funding opportunit


---

## **Parse Research Papers**

This cell iterates over all PDF files discovered in the `Papers` subfolder and extracts their text using the same `extract_pdf_text()` and `chunk_text()` functions defined in Cell 2. Papers that yield no extractable text — typically scanned or image-based PDFs that lack a text layer — are skipped with a warning rather than causing a pipeline failure. Successfully parsed papers are stored in the `papers` dictionary keyed by filename, with each entry holding the full text, chunked segments, and file path for downstream indexing and retrieval.

In [18]:
papers = {}

for p in PAPER_PATHS:
    print(f"Parsing: {p.name} ...", end=" ")
    txt = extract_pdf_text(p)
    if not txt.strip():
        print(f"⚠ SKIPPED (no extractable text — likely scanned/image PDF)")
        continue
    cks = chunk_text(txt)
    if not cks:
        print(f"⚠ SKIPPED (no chunks produced)")
        continue
    papers[p.name] = {"text": txt, "chunks": cks, "path": str(p)}
    print(f"{len(txt):,} chars → {len(cks)} chunks")

print(f"\nSuccessfully parsed: {len(papers)}/{len(PAPER_PATHS)} papers")

for name, data in papers.items():
    preview = data["text"][:300].replace("\n", " ")
    print(f"\n--- {name} ---")
    print(preview)

Parsing: 2021_EPJ_MVMCInfoOps.pdf ... 91,148 chars → 31 chunks
Parsing: 2024_ICWSM_Data_Challenge__Post_API_Data_Collection.pdf ... 13,978 chars → 4 chunks
Parsing: 23-US-DHS-001.pdf ... 101,995 chars → 46 chunks
Parsing: AAAI IAA CV.pdf ... 40,482 chars → 13 chunks
Parsing: Acquiring Maintainable AI_Enable Systems_Final.pdf ... 28,934 chars → 13 chunks
Parsing: ALL18.pdf ... 25,432 chars → 12 chunks
Parsing: Analysis_of_Malware_Communities_Using_Multi_Modal_Features.pdf ... 68,983 chars → 22 chunks
Parsing: Arrow White Paper DExTra.pdf ... 17,304 chars → 8 chunks
Parsing: A_Complex_Network_Approach_to_Find_Latent_Terorrist_Communities.pdf ... 69,172 chars → 22 chunks
Parsing: Benson_MA491_NLP.pdf ... 57,433 chars → 15 chunks
Parsing: BotBuster___AAAI.pdf ... 49,132 chars → 14 chunks
Parsing: CausalOrgInorgContent.pdf ... 27,491 chars → 14 chunks
Parsing: Characterizing_Communities_of_Hashtag_Usage_on_Twitter_During_the_2020_COVID_19_Pandemic.pdf ... 113,785 chars → 42 chunks
Parsing: 

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


20,039 chars → 17 chunks
Parsing: Take_boards.pdf ... 599 chars → 1 chunks
Parsing: Text Analysis Using Automated Language Translators.pdf ... 2,004 chars → 1 chunks
Parsing: The ABCs of AI-Enabled Intelligence Analysis - War on the Rocks.pdf ... 

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


20,034 chars → 9 chunks
Parsing: TrainingSetSize.pdf ... 40,015 chars → 19 chunks
Parsing: Tweets-to-touchdowns.pdf ... 25,574 chars → 12 chunks
Parsing: Unobtrusive Email.pdf ... 32,416 chars → 16 chunks
Parsing: Utility Seeking in Complex Social Systems.pdf ... 46,855 chars → 21 chunks
Parsing: Vol33Iss1_INSNApdf.pdf ... 75,999 chars → 17 chunks
Parsing: Vulnerable_Code_Detection.pdf ... 33,263 chars → 12 chunks
Parsing: WEIRD.pdf ... 59,694 chars → 16 chunks
Parsing: White Paper Brain Gaze.pdf ... 22,043 chars → 11 chunks
Parsing: YouTube-COVID.pdf ... 34,892 chars → 14 chunks

Successfully parsed: 110/112 papers

--- 2021_EPJ_MVMCInfoOps.pdf ---
Uyhengetal. RESEARCH Mapping State-Sponsored Information Operations with Multi-View Modularity Clustering Joshua Uyheng*, Iain J. Cruickshank and Kathleen M. Carley *Correspondence: juyheng@cs.cmu.edu Abstract CASOSCenter,Institutefor This paper presents a new computational framework for mapping sta

--- 2024_ICWSM_Data_Challenge__Post_API_

---

## **Build FAISS Vector Index**

This cell converts all parsed text chunks into numerical vector representations and loads them into FAISS indices for fast similarity search. A separate `IndexFlatIP` (inner-product) index is built for the NOFO and for each individual research paper, where inner-product on unit-normalized vectors is equivalent to cosine similarity. Papers that produce empty embeddings — which can occur if a parsed document yielded only whitespace or unparseable content — are skipped with a warning to prevent downstream retrieval errors.

In [19]:
DIM = 1536

# ── Build NOFO index ──────────────────────────────────────────────────────────
print("Embedding NOFO chunks ...")
nofo_vecs  = embed_batch(nofo_chunks)
nofo_index = faiss.IndexFlatIP(DIM)  # Inner-product = cosine on unit vectors
nofo_index.add(nofo_vecs)
print(f"  NOFO index: {nofo_index.ntotal} vectors")

# ── Build per-paper indices ───────────────────────────────────────────────────
print("\nEmbedding paper chunks ...")
paper_indices = {}

for name, data in papers.items():
    vecs = embed_batch(data["chunks"])
    if vecs.shape[0] == 0:
        print(f"  ⚠ SKIPPED {name} (empty embedding)")
        continue
    idx = faiss.IndexFlatIP(DIM)
    idx.add(vecs)
    paper_indices[name] = idx
    print(f"  {name}: {idx.ntotal} vectors")

print("\nAll indices built.")

Embedding NOFO chunks ...
  NOFO index: 32 vectors

Embedding paper chunks ...
  2021_EPJ_MVMCInfoOps.pdf: 31 vectors
  2024_ICWSM_Data_Challenge__Post_API_Data_Collection.pdf: 4 vectors
  23-US-DHS-001.pdf: 46 vectors
  AAAI IAA CV.pdf: 13 vectors
  Acquiring Maintainable AI_Enable Systems_Final.pdf: 13 vectors
  ALL18.pdf: 12 vectors
  Analysis_of_Malware_Communities_Using_Multi_Modal_Features.pdf: 22 vectors
  Arrow White Paper DExTra.pdf: 8 vectors
  A_Complex_Network_Approach_to_Find_Latent_Terorrist_Communities.pdf: 22 vectors
  Benson_MA491_NLP.pdf: 15 vectors
  BotBuster___AAAI.pdf: 14 vectors
  CausalOrgInorgContent.pdf: 14 vectors
  Characterizing_Communities_of_Hashtag_Usage_on_Twitter_During_the_2020_COVID_19_Pandemic.pdf: 42 vectors
  Chat GPT Bias final w copyright.pdf: 12 vectors
  ClassifiersCrowdSource.pdf: 4 vectors
  Clustering_Analysis_of_Website_Usage_on_Twitter_during_the_COVID_19_Pandemic.pdf: 16 vectors
  Cohort_Optimization_Methods_SNAMS_2021_working_draft (4).

---

## **Filter Papers by Topical Relevance to NOFO**

This cell scores each parsed paper against a query string summarizing the core themes of PAR-25-136, using cosine similarity between hash-projection embeddings as a proxy for topical relevance. `retrieve()` returns the top-k most similar chunks from a given FAISS index for a query, and `score_paper_relevance()` averages those similarity scores across the top-k chunks to produce a single relevance score per paper. Papers scoring at or above `RELEVANCE_THRESHOLD` are retained in `relevant_papers` for use in downstream context retrieval — papers below the threshold are filtered out to reduce noise in the proposal generation prompts.

In [20]:
def retrieve(query: str, index: faiss.IndexFlatIP,
             chunks: List[str], top_k: int = 5) -> List[Tuple[float, str]]:
    """Return (score, chunk) pairs for the top-k most similar chunks."""
    q_vec = get_embedding(query).reshape(1, -1)
    scores, ids = index.search(q_vec, top_k)
    return [(float(scores[0][i]), chunks[ids[0][i]])
            for i in range(top_k) if ids[0][i] >= 0]

def score_paper_relevance(paper_name: str, nofo_summary_query: str,
                          top_k: int = 10) -> float:
    """
    Score a paper's relevance to the NOFO by:
    1. Retrieving top-k paper chunks matching the NOFO query
    2. Averaging the cosine similarity scores
    """
    results = retrieve(nofo_summary_query,
                       paper_indices[paper_name],
                       papers[paper_name]["chunks"],
                       top_k=top_k)
    if not results:
        return 0.0
    return float(np.mean([s for s, _ in results]))

# ── Core NOFO themes as query string ─────────────────────────────────────────
NOFO_QUERY = (
    "digital mental health interventions mobile health smartphone wearable "
    "sensor adaptive intervention clinical trial evidence-based platform "
    "engagement adherence health disparities behavior change machine learning "
    "predictive analytics real-time data collection population mental health"
)

print("Scoring paper relevance to NOFO ...")
print(f"Query: '{NOFO_QUERY[:80]}...'\n")

relevance_scores = {}
for name in papers:
    score = score_paper_relevance(name, NOFO_QUERY)
    relevance_scores[name] = score
    status = "✓ RELEVANT" if score >= RELEVANCE_THRESHOLD else "✗ LOW RELEVANCE"
    print(f"  {name[:50]:<50}  score={score:.4f}  {status}")

relevant_papers = {k: v for k, v in papers.items()
                   if relevance_scores[k] >= RELEVANCE_THRESHOLD}

print(f"\nPapers passing threshold ({RELEVANCE_THRESHOLD}): {len(relevant_papers)}/{len(papers)}")

Scoring paper relevance to NOFO ...
Query: 'digital mental health interventions mobile health smartphone wearable sensor ada...'

  2021_EPJ_MVMCInfoOps.pdf                            score=0.4624  ✓ RELEVANT
  2024_ICWSM_Data_Challenge__Post_API_Data_Collectio  score=0.4345  ✗ LOW RELEVANCE
  23-US-DHS-001.pdf                                   score=0.4599  ✓ RELEVANT
  AAAI IAA CV.pdf                                     score=0.4135  ✗ LOW RELEVANCE
  Acquiring Maintainable AI_Enable Systems_Final.pdf  score=0.3777  ✗ LOW RELEVANCE
  ALL18.pdf                                           score=0.3531  ✗ LOW RELEVANCE
  Analysis_of_Malware_Communities_Using_Multi_Modal_  score=0.3880  ✗ LOW RELEVANCE
  Arrow White Paper DExTra.pdf                        score=0.4148  ✗ LOW RELEVANCE
  A_Complex_Network_Approach_to_Find_Latent_Terorris  score=0.4239  ✗ LOW RELEVANCE
  Benson_MA491_NLP.pdf                                score=0.4544  ✓ RELEVANT
  BotBuster___AAAI.pdf                       

---

## **Generate Bridging Research Ideas**

This cell defines two RAG context-building functions and uses them to generate five novel research ideas that bridge the PI's existing expertise with the requirements of PAR-25-136. `build_nofo_context()` retrieves the most relevant NOFO chunks for a given query, and `build_paper_context()` retrieves and ranks the most relevant chunks across all filtered papers. These two context strings are injected into a structured LLM prompt that asks the model to identify specific transferable methods from the PI's past work — such as multi-agent simulation, network analysis, and ML classification — and propose concrete digital mental health interventions grounded in those methods. The resulting ideas feed directly into the proposal generation step.

In [21]:
def build_nofo_context(query: str, top_k: int = TOP_K_NOFO) -> str:
    """Retrieve and concatenate the top-k most relevant NOFO chunks for a query."""
    results = retrieve(query, nofo_index, nofo_chunks, top_k=top_k)
    return "\n\n---\n\n".join(chunk for _, chunk in results)

def build_paper_context(query: str, top_k: int = TOP_K_PAPERS) -> str:
    """Retrieve, rank, and concatenate the top paper chunks across all relevant papers."""
    all_results = []
    for name, data in relevant_papers.items():
        results = retrieve(query, paper_indices[name], data["chunks"], top_k=top_k)
        for score, chunk in results:
            all_results.append((score, name, chunk))
    all_results.sort(key=lambda x: -x[0])
    sections = []
    for score, name, chunk in all_results[:top_k * 2]:
        sections.append(f"[From: {name} | score={score:.3f}]\n{chunk}")
    return "\n\n---\n\n".join(sections)

# ── Build context for idea generation ────────────────────────────────────────
IDEAS_QUERY = (
    "transferable methods behavioral science network analysis machine learning "
    "real-time adaptive intervention mental health digital platform engagement"
)

nofo_ctx   = build_nofo_context(IDEAS_QUERY)
papers_ctx = build_paper_context(IDEAS_QUERY)

IDEA_SYSTEM = """You are a senior NIH grant strategist and computational behavioral \
scientist. Your task is to identify novel research ideas that bridge the PI's existing \
expertise in network science, multi-agent simulation, machine learning, and behavioral \
modeling with the requirements of an NIH R01 NOFO focused on digital mental health \
interventions. Be specific, creative, and scientifically rigorous. All ideas must be \
responsive to the NOFO."""

IDEA_PROMPT = f"""NOFO CONTEXT (PAR-25-136 — Laboratories to Optimize Digital Health):
{nofo_ctx}

PI's PAST WORK / RESEARCH REPOSITORY EXCERPTS:
{papers_ctx}

TASK:
Generate 5 specific, innovative research ideas that:
1. Draw directly on methods or findings from the PI's past work
2. Are fully responsive to PAR-25-136 (digital mental health, clinical trial required,
   evidence-based platform with 1000+ users, mechanism specification required)
3. Propose a concrete digital mental health intervention
4. Identify transferable technical capabilities (e.g., multi-agent simulation →
   adaptive intervention optimization; network analysis → social support mapping;
   ML classification → real-time clinical state detection)

Format each idea as:
IDEA N: [Title]
- Core Concept: ...
- Transferable Method from Past Work: ...
- Target Population & Platform: ...
- Proposed Mechanism: ...
- Innovation: ...
"""

print("Generating research ideas via LLM ...\n")
research_ideas = llm_complete(
    messages=[{"role": "user", "content": IDEA_PROMPT}],
    system=IDEA_SYSTEM,
    max_tokens=1500
)
print(research_ideas)

Generating research ideas via LLM ...

IDEA 1: "Real-Time Social Support Mapping for Mental Health Engagement"

- Core Concept: Utilize network analysis to map and enhance peer support interactions within a digital mental health platform to boost engagement and adherence.
- Transferable Method from Past Work: Leverage the PI's expertise in social network analysis to identify key actors and sub-networks that promote engagement.
- Target Population & Platform: Adults with mild to moderate anxiety using an established mental health app with a minimum of 1000 users.
- Proposed Mechanism: Integrating real-time network mapping features to identify support needs and suggest engagement with peers or professionals within the app.
- Innovation: Providing personalized social support recommendations based on real-time user interaction data, potentially improving adherence and outcomes.

IDEA 2: "Adaptive Multi-Agent System for Anxiety Management"

- Core Concept: Develop a multi-agent system to dy

---

## **Generate NIH Technical Approach**

This cell constructs the full Technical Approach draft in two sequential LLM calls to avoid context window constraints. Part 1 generates the Significance and Innovation sections (minimum 700 words combined), grounding the proposal in the epidemiological burden, limitations of current digital health approaches, and the PI's transferable methodological innovations. Part 2 generates the Approach section across five required subsections (3a–3e, minimum 1,100 words), covering the intervention mechanism, target population, study design, analytic strategy, and 4-year timeline. The two parts are combined into a single draft and checked against the word count target — if the combined draft falls below 1,800 words, an automatic expansion pass rewrites and extends the draft before it proceeds to the evaluation loop.

In [22]:
# ── Retrieve NOFO-specific guidance ──────────────────────────────────────────
PROPOSAL_QUERY = (
    "significance innovation approach clinical trial digital mental health "
    "mechanism target engagement adaptive intervention platform users "
    "health disparities evaluation criteria review"
)
nofo_proposal_ctx   = build_nofo_context(PROPOSAL_QUERY, top_k=TOP_K_NOFO)
papers_proposal_ctx = build_paper_context(PROPOSAL_QUERY, top_k=TOP_K_PAPERS)

PROPOSAL_SYSTEM = """You are a senior NIH grant writer with 20+ years of experience \
writing funded R01 proposals for NIMH. You write exclusively in polished, professional \
NIH proposal prose. Your responses are always thorough, fully developed, and meet \
explicit word count requirements without exception. You never truncate, summarize, \
or use placeholder text. Every section you write is complete, substantive, and ready \
for submission."""

# ── Part 1: Significance + Innovation ────────────────────────────────────────
PART1_PROMPT = f"""You are writing the Significance and Innovation sections of an NIH \
R01 Technical Approach in response to PAR-25-136 (Laboratories to Optimize Digital \
Health, NIMH).

NOFO REQUIREMENTS — read carefully and address every point:
{nofo_proposal_ctx}

PROPOSED RESEARCH IDEAS TO DRAW FROM:
{research_ideas[:2000]}

INSTRUCTIONS:
Write ONLY the Significance and Innovation sections. Do not write the Approach section.
These two sections combined MUST be at least 700 words. Do not stop writing until you \
have fully developed every point below.

1. SIGNIFICANCE (minimum 350 words, written as flowing prose across at least 4 paragraphs)
   Paragraph 1 — Epidemiological burden: Establish the scope of the mental health crisis \
in the US, citing specific prevalence statistics for the target condition. Quantify the \
treatment gap and explain why existing solutions are insufficient.
   Paragraph 2 — Limitations of current digital health approaches: Describe what NIMH \
has funded to date, what stage of research most prior work has reached, and what critical \
gap in effectiveness, reach, or engagement remains unaddressed.
   Paragraph 3 — Impact of the proposed intervention: Justify the anticipated effect size \
with reference to prior efficacy data or mechanism-outcome associations. Explain the \
clinical meaningfulness of the expected improvement compared to existing approaches.
   Paragraph 4 — Scalability and dissemination: Explain how the intervention could be \
deployed at scale given real-world constraints including provider availability, healthcare \
financing, and typical service use patterns. Address health disparities explicitly — \
identify which racial, ethnic, or socioeconomic populations are disproportionately \
affected and how this proposal addresses that gap.

2. INNOVATION (minimum 350 words, written as flowing prose across at least 3 paragraphs)
   Paragraph 1 — What makes this novel: Describe specifically how this proposal departs \
from the current state of the science. Identify at least two concrete methodological or \
conceptual innovations not present in prior NIMH-funded work.
   Paragraph 2 — Platform and technology capabilities: Explain how the proposed platform \
capitalizes on unique mobile and wireless functionality, real-time data collection, \
software update cycles informed by user feedback, and interoperability with existing \
health information systems.
   Paragraph 3 — Methodological innovations from PI's past work: Identify specific \
methods from the PI's prior research (network analysis, multi-agent simulation, ML \
classification, behavioral modeling) and explain precisely how each transfers to the \
digital mental health context. Be concrete — name the method, name the prior application, \
and describe the translation.

Write in formal NIH proposal prose throughout. No bullet points. No headers beyond \
the two section titles. No placeholder text. Write every paragraph in full."""

print("Generating Significance + Innovation (Part 1) ...\n")
part1 = llm_complete(
    messages=[{"role": "user", "content": PART1_PROMPT}],
    system=PROPOSAL_SYSTEM,
    max_tokens=2000
)
print(part1)
print(f"\nPart 1 word count: ~{len(part1.split()):,} words")

# ── Part 2: Approach ──────────────────────────────────────────────────────────
PART2_PROMPT = f"""You are writing the Approach section of an NIH R01 Technical \
Approach in response to PAR-25-136 (Laboratories to Optimize Digital Health, NIMH).

NOFO REQUIREMENTS — read carefully and address every point:
{nofo_proposal_ctx}

PI's PAST WORK (relevant excerpts to draw methods and preliminary evidence from):
{papers_proposal_ctx}

INSTRUCTIONS:
Write ONLY the Approach section (subsections 3a through 3e). Do not write Significance \
or Innovation. This section MUST be at least 1,100 words total. Do not stop writing \
until every subsection is fully developed as specified below.

3. APPROACH

3a. Conceptual Framework and Intervention Mechanism (minimum 220 words, flowing prose)
   - Open with a clear statement of the target mechanism — the specific psychological, \
behavioral, or neurological process the intervention is designed to engage.
   - Provide the theoretical basis: name the behavioral science theory (e.g., Social \
Cognitive Theory, Self-Determination Theory, Ecological Momentary Assessment framework) \
and explain the empirical evidence linking that theory to the clinical outcome of interest.
   - Describe how the digital platform will engage the target mechanism in practice. \
Explain the logic model: if the mechanism is engaged, how does that produce the \
hypothesized change in clinical outcomes?
   - Explain how you will measure mechanism engagement — what assessments or passive \
sensing approaches will confirm the mechanism was activated.

3b. Target Population and Platform (minimum 200 words, flowing prose)
   - Describe the target population with specificity: age range, clinical inclusion \
criteria (symptom threshold on a validated scale), and exclusion criteria.
   - Name the digital platform and describe it concretely: what type of intervention \
it delivers, its existing evidence base, and confirm it has at least 1,000 active daily \
users. If hypothetical, describe it as if real.
   - Describe the enrollment process in detail: how current users will be identified, \
how consent will be obtained digitally, and how participants will be randomized. \
Specify the randomization method (e.g., stratified block randomization) and allocation ratio.

3c. Study Design and Specific Aims (minimum 220 words)
   - State 3 Specific Aims clearly and completely. Each Aim should have a one-sentence \
hypothesis and identify the primary analysis associated with it.
   - Identify primary and secondary outcome measures. Name the validated instruments \
(e.g., PHQ-9, GAD-7, PROMIS) and justify their use.
   - Provide a sample size and power analysis: state the assumed effect size, alpha, \
power, estimated attrition, and resulting required N per arm. Justify the effect size \
assumption with reference to prior literature.

3d. Analytic Strategy (minimum 220 words, flowing prose)
   - Primary effectiveness analysis: specify the statistical model (e.g., mixed-effects \
linear model, intent-to-treat), covariates, and how missing data will be handled.
   - Mechanism engagement analysis: describe how you will test whether the intervention \
engaged the target mechanism and whether mechanism engagement mediates outcomes. \
Name the analytic approach (e.g., causal mediation analysis).
   - Machine learning and adaptive components: describe the ML approach used to \
personalize or adapt the intervention (e.g., reinforcement learning, classification \
models). Explain what data inputs the model uses, what it outputs, and how its \
recommendations are delivered to users in real time.

3e. Timeline and Feasibility (minimum 200 words, flowing prose — no bullet points)
   - Describe the 4-year project timeline as a narrative, covering Year 1 through \
Year 4. For each year, describe the major activities, milestones, and what will be \
delivered or completed.
   - Address feasibility explicitly: explain why the platform partnership is in place, \
why the target enrollment is achievable, and what preliminary evidence or prior \
experience supports the team's ability to execute this work.

Write every subsection in full. Use formal NIH proposal prose throughout. \
Subsection 3c may use a structured format for the Specific Aims but all other \
subsections must be written as flowing prose paragraphs. No placeholder text."""

print("\nGenerating Approach (Part 2) ...\n")
part2 = llm_complete(
    messages=[{"role": "user", "content": PART2_PROMPT}],
    system=PROPOSAL_SYSTEM,
    max_tokens=2500
)
print(part2)
print(f"\nPart 2 word count: ~{len(part2.split()):,} words")

# ── Combine ───────────────────────────────────────────────────────────────────
proposal_draft_v1 = part1 + "\n\n" + part2
combined_wc = len(proposal_draft_v1.split())
print(f"\nCombined draft word count: ~{combined_wc:,} words")

# ── Expansion pass if under target ───────────────────────────────────────────
if combined_wc < 1800:
    print(f"\n⚠ Draft is {1800 - combined_wc} words short of target. Running expansion pass ...")

    EXPAND_PROMPT = f"""The following NIH R01 Technical Approach draft is too short at \
~{combined_wc} words. It must reach at least 1,800 words for submission.

CURRENT DRAFT:
{proposal_draft_v1}

TASK: Rewrite the entire draft, expanding every section to its full required length.
- Significance: minimum 350 words across 4 paragraphs
- Innovation: minimum 350 words across 3 paragraphs
- Each Approach subsection (3a-3e): minimum 200 words each
- Total combined output: minimum 1,800 words

Do not add new sections. Do not change the structure. Expand the existing content \
by adding supporting evidence, mechanistic detail, methodological specifics, and \
clinical justification to every paragraph. Write in formal NIH prose throughout. \
You MUST produce at least 1,800 words. Do not stop writing until the full draft \
is complete."""

    proposal_draft_v1 = llm_complete(
        messages=[{"role": "user", "content": EXPAND_PROMPT}],
        system=PROPOSAL_SYSTEM,
        max_tokens=4000
    )
    expanded_wc = len(proposal_draft_v1.split())
    print(f"Expanded draft word count: ~{expanded_wc:,} words")
    if expanded_wc < 1600:
        print(f"  ⚠ Still below target after expansion ({expanded_wc} words). "
              f"The eval loop reviser will attempt further expansion.")
else:
    print("✓ Draft meets word count target.")

# ── Confirm what enters the eval loop ────────────────────────────────────────
print(f"\nDraft entering eval loop: ~{len(proposal_draft_v1.split()):,} words")
print("="*60)

Generating Significance + Innovation (Part 1) ...

**SIGNIFICANCE**

The mental health crisis in the United States continues to pose a substantial public health challenge, deeply affecting individuals and communities across the country. Epidemiological studies indicate that approximately 20% of adults and 30% of adolescents experience symptoms of mental disorders annually, with anxiety disorders being among the most prevalent (Kessler et al., 2005). Despite this high prevalence, only about half of these individuals receive treatment, highlighting a significant treatment gap (Wang et al., 2005). This disparity is exacerbated by factors such as stigma, geographic barriers, and shortages of mental health professionals. Current interventions, largely dependent on traditional face-to-face modalities, are insufficient in meeting the overwhelming demand, necessitating innovative solutions to bridge this gap and make care more accessible to those in need.

In recent years, NIMH has directed su

---

## **LLM Self-Evaluation Loop**

This cell implements an automated review-and-revision loop that simulates NIH study section feedback before the draft reaches human review. `evaluate_proposal()` scores the draft against PAR-25-136 review criteria across three factors (Significance & Innovation, Rigor & Feasibility, and Expertise & Resources) and returns a structured verdict of either MEETS CRITERIA or NEEDS REVISION. `revise_proposal()` rewrites the draft in response to that feedback while enforcing a minimum word count of 1,700 words to prevent the reviser from collapsing the draft. The loop runs for up to `MAX_EVAL_LOOPS` rounds, stopping early if the draft is approved — if the draft still falls below 1,600 words after all revision rounds, a final emergency expansion pass is triggered before handing off to human review.

In [23]:
EVAL_SYSTEM = """You are a senior NIH study section reviewer for NIMH digital mental \
health R01 proposals. Evaluate the proposal against PAR-25-136 review criteria \
rigorously and objectively. Provide structured, actionable feedback. Be specific \
about deficiencies and improvements."""

NIH_CRITERIA = """
NIH REVIEW CRITERIA (PAR-25-136):
Factor 1 – Significance & Innovation:
  - Is the proposed digital mental health intervention important and innovative?
  - Is effect size empirically justified?
  - Is the intervention scalable and disseminatable?
  - Does it leverage unique mobile/wearable/platform functionality?

Factor 2 – Rigor & Feasibility (Approach):
  - Are specific Aims clearly stated?
  - Is the target mechanism explicitly specified AND is there a plan to assess
    mechanism engagement?
  - Is the platform well-established with ≥1000 active users?
  - Is enrollment/randomization feasible?
  - Is sample size/power justified?
  - Are outcome measures validated?
  - Does the design go beyond the primary effectiveness trial?

Factor 3 – Expertise & Resources:
  - Do investigators have relevant expertise?
  - Are stakeholder partnerships described?

NON-RESPONSIVE flags (immediate rejection):
  - No clinical trial proposed
  - Platform has <1000 active users
  - No mechanism specification
  - Intervention not empirically based
  - No active comparator

STANDING QUALITY STANDARDS (must be met before APPROVED verdict):
  Significance:
  - Minimum 8 in-text citations; every major epidemiological or economic claim
    must be attributed to a specific source (author/year or agency)
  - Health disparities paragraph must name at least two specific populations
    (e.g., Black adults, rural communities, uninsured) with population-specific
    statistics, not generic language
  Innovation:
  - Any named computational method (simulation, network analysis, ML/RL) must
    be operationalized: define inputs, outputs, and how results feed back into
    intervention logic — vague mentions are insufficient
  Approach:
  - Adaptive/RL algorithm must explicitly define state space, action space,
    and reward function
  - Sample size justification must state Cohen's d, alpha, power target, and
    assumed attrition rate — a number alone is insufficient
  - Each Specific Aim must include a contingency or alternative approach if the
    primary method fails
  - Year 1 timeline must list monthly milestones with concrete deliverables
    (e.g., Month 3: IRB approval; Month 6: 50 participants enrolled)
  - Must include a Data Safety / IRB paragraph covering de-identification
    protocol, participant withdrawal, and HIPAA compliance
  Format:
  - Minimum 1,800 words; target density consistent with a 5-page NIH submission
  - All body text in formal NIH prose; no bullet points within proposal sections
"""

def evaluate_proposal(draft: str) -> Tuple[str, bool, str]:
    """
    Evaluates the proposal draft against NIH PAR-25-136 criteria.
    Returns:
        evaluation_text : full reviewer feedback
        meets_criteria  : True if proposal is approvable as-is
        summary_verdict : short pass/fail string
    """
    eval_prompt = f"""{NIH_CRITERIA}

PROPOSAL DRAFT TO EVALUATE:
{draft}

Provide your evaluation in this exact format:

OVERALL VERDICT: [MEETS CRITERIA / NEEDS REVISION]

FACTOR 1 – SIGNIFICANCE & INNOVATION: [STRONG / ADEQUATE / WEAK]
Comments: ...

FACTOR 2 – RIGOR & FEASIBILITY: [STRONG / ADEQUATE / WEAK]
Comments: ...

FACTOR 3 – EXPERTISE & RESOURCES: [STRONG / ADEQUATE / WEAK]
Comments: ...

NON-RESPONSIVE FLAGS: [NONE / LIST ANY]

PRIORITY REVISIONS REQUIRED (if any):
1. ...
2. ...
3. ...

SUGGESTED IMPROVEMENTS:
..."""

    evaluation = llm_complete(
        messages=[{"role": "user", "content": eval_prompt}],
        system=EVAL_SYSTEM,
        max_tokens=1000
    )
    meets = "MEETS CRITERIA" in evaluation.upper()
    verdict = "APPROVED" if meets else "NEEDS REVISION"
    return evaluation, meets, verdict

def revise_proposal(draft: str, evaluation: str) -> str:
    """
    Revises the proposal based on reviewer feedback while preserving word count.
    """
    current_wc = len(draft.split())
    revision_prompt = f"""You are revising an NIH R01 Technical Approach based on \
study section feedback. The current draft is approximately {current_wc} words. \
Your revised version MUST be at least 1,700 words. Do not shorten, summarize, \
or truncate any section — only improve and expand.

CURRENT DRAFT:
{draft}

REVIEWER FEEDBACK:
{evaluation}

NOFO KEY REQUIREMENTS (PAR-25-136):
{nofo_proposal_ctx[:1200]}

STANDING QUALITY STANDARDS THE NEXT REVIEWER WILL CHECK:
{NIH_CRITERIA}

TASK: Rewrite the Technical Approach addressing ALL reviewer concerns while:
- Preserving and expanding strong sections — do not delete content that is working
- Satisfying every item in STANDING QUALITY STANDARDS above
- Maintaining a minimum of 1,700 words in the final output — this is non-negotiable
- Writing every section in full NIH proposal prose
- Not truncating any subsection for any reason
- Ending only when the full revised proposal is complete

You MUST produce at least 1,700 words. Do not stop writing until every section \
is fully developed and the complete proposal has been written."""

    revised = llm_complete(
        messages=[{"role": "user", "content": revision_prompt}],
        system=PROPOSAL_SYSTEM,
        max_tokens=4000
    )
    return revised

# ── Run evaluation loop ───────────────────────────────────────────────────────
current_draft   = proposal_draft_v1
all_evaluations = []
approved        = False

print(f"Starting eval loop with draft of ~{len(current_draft.split()):,} words\n")

for loop in range(1, MAX_EVAL_LOOPS + 1):
    print(f"\n{'='*60}")
    print(f"LLM EVALUATION — Round {loop}/{MAX_EVAL_LOOPS}")
    print(f"Draft word count entering this round: ~{len(current_draft.split()):,} words")
    print('='*60)

    evaluation, meets, verdict = evaluate_proposal(current_draft)
    all_evaluations.append({"round": loop, "evaluation": evaluation,
                             "verdict": verdict})

    print(evaluation)
    print(f"\n→ Verdict: {verdict}")

    if meets:
        print("\n✓ Proposal APPROVED by LLM evaluator. Proceeding to human review.")
        approved = True
        break

    if loop < MAX_EVAL_LOOPS:
        print(f"\nRevising proposal (round {loop}) ...")
        current_draft = revise_proposal(current_draft, evaluation)
        revised_wc = len(current_draft.split())
        print(f"Revision {loop} complete (~{revised_wc:,} words).")
        if revised_wc < 1600:
            print(f"  ⚠ WARNING: Revision collapsed word count to {revised_wc} words. "
                  f"Check reviser max_tokens and prompt enforcement.")
    else:
        print(f"\nMax evaluation loops ({MAX_EVAL_LOOPS}) reached.")
        print("Proceeding to human review with best available draft.")

# ── Post-loop word count guard ────────────────────────────────────────────────
final_llm_draft = current_draft
final_wc        = len(final_llm_draft.split())

print(f"\nFinal LLM-evaluated draft word count: ~{final_wc:,} words")

if final_wc < 1600:
    print(f"\n⚠ Final draft is only {final_wc} words — running emergency expansion ...")
    emergency_prompt = f"""The following NIH R01 Technical Approach is too short at \
~{final_wc} words and must be expanded to at least 1,700 words before human review.

CURRENT DRAFT:
{final_llm_draft}

TASK: Expand every section of this draft without changing its structure or adding \
new sections. Add supporting evidence, mechanistic detail, methodological specifics, \
and clinical justification to each paragraph until the full draft reaches at least \
1,700 words. Write in formal NIH prose throughout. Do not stop until complete."""

    final_llm_draft = llm_complete(
        messages=[{"role": "user", "content": emergency_prompt}],
        system=PROPOSAL_SYSTEM,
        max_tokens=4000
    )
    print(f"Emergency expansion complete: ~{len(final_llm_draft.split()):,} words")

print("\nFinal LLM-evaluated draft ready for human review.")

Starting eval loop with draft of ~2,082 words


LLM EVALUATION — Round 1/3
Draft word count entering this round: ~2,082 words
OVERALL VERDICT: NEEDS REVISION

FACTOR 1 – SIGNIFICANCE & INNOVATION: ADEQUATE  
Comments: The proposal addresses a significant gap in mental health care access and proposes an innovative approach using real-time social network analysis. However, the innovation section lacks detailed operationalization of computational methods, including how the adaptive multi-agent system leverages machine learning or reinforcement learning principles. More explicit definitions of inputs, outputs, and feedback loops within these computational frameworks are necessary for the innovation claim to be fully substantiated.

FACTOR 2 – RIGOR & FEASIBILITY: ADEQUATE  
Comments: While the specific aims and a detailed framework are presented, the proposal needs more elaboration on the alternative strategies if primary methods fail for each specific aim. The sample size justification is

---

## **Human Evaluation Checkpoint**

This cell implements the human review gate, whose behavior is controlled by `RUN_MODE` set in Cell 0. In `"auto"` mode the checkpoint is bypassed entirely, `HUMAN_APPROVED` is set to `True` automatically, and a 500-word preview of the autonomous draft is printed for reference. In `"human"` mode the full draft is printed for review and the pipeline pauses — the reviewer reads the draft, adds specific editorial feedback to `HUMAN_FEEDBACK`, and either sets `HUMAN_APPROVED = True` to accept the draft as-is or runs Cell 10 to incorporate the feedback into a revised version. This checkpoint is where domain expertise, citation accuracy, and NIH-specific editorial judgment are applied before the final proposal is saved and exported.

In [24]:
# ── Human Evaluation Checkpoint ──────────────────────────────────────────────
if RUN_MODE == "auto":
    print("="*60)
    print("RUN MODE: AUTO — skipping human review checkpoint")
    print("="*60)
    print("\nAutonomous draft is final. Set RUN_MODE = 'human' in Cell 0")
    print("and re-run the notebook to incorporate human feedback.\n")
    HUMAN_APPROVED = True  # auto-approve to allow pipeline to complete
    auto_final_draft = final_llm_draft
    print(f"Auto draft word count : ~{len(auto_final_draft.split()):,} words")
    print("\nAuto draft preview (first 500 words):")
    print("-"*60)
    print(" ".join(auto_final_draft.split()[:500]) + " ...")
    print("-"*60)

else:
    print("="*60)
    print("RUN MODE: HUMAN — manual review checkpoint active")
    print("="*60)
    print()
    print("Review the draft below. When satisfied:")
    print("  1. Set HUMAN_APPROVED = True")
    print("  2. Or add feedback to HUMAN_FEEDBACK and run Cell 10")
    print()
    print("-"*60)
    print(final_llm_draft)
    print("-"*60)

    HUMAN_APPROVED = False  # set to True once satisfied

    HUMAN_FEEDBACK = """
    1. SIGNIFICANCE — Add citations throughout. Each major claim (prevalence rates,
    treatment gap, economic burden, digital intervention efficacy) needs a specific
    reference. Target at least 8-10 citations in this section alone.

    2. SIGNIFICANCE — Health disparities paragraph is too generic. Name specific
    populations (e.g., Black and Hispanic adults, rural communities, uninsured/
    underinsured individuals) and include prevalence or access statistics specific
    to those groups.

    3. INNOVATION — Operationalize the multi-agent simulation and network analysis
    components. Explain concretely: what does the simulation model, what are the
    network nodes/edges, how does analysis output feed into the intervention logic?

    4. APPROACH (3a) — Describe the reinforcement learning algorithm in technical
    detail: define the state space (symptom scores, usage patterns), action space
    (content modules, notification timing), and reward function (PHQ-9 trajectory,
    engagement rate).

    5. APPROACH (3c) — Expand sample size justification. State the assumed effect
    size (Cohen's d), alpha level (0.05), power target (0.80), and expected
    attrition rate used in the power calculation.

    6. APPROACH (3d) — Add a Risks and Alternatives section. For each major
    methodological risk (e.g., high attrition, platform dropout, algorithm
    underperformance), provide a specific mitigation strategy or alternative
    analytical approach.

    7. APPROACH (3e) — Year 1 timeline needs concrete milestones with target months.
    Example: Month 3 — IRB approval; Month 4 — platform integration complete;
    Month 6 — first 50 participants enrolled; Month 12 — baseline data collection
    closed.

    8. APPROACH — Add a brief Data Safety and IRB section covering: IRB oversight
    plan, data de-identification protocol, participant withdrawal procedures, and
    HIPAA compliance for app-collected health data.

    9. LENGTH — Current draft should target approximately 5 pages (~2,000 words).
    Prioritize expanding Approach subsections 3a, 3c, and 3e with additional
    mechanistic detail, and add a Literature Context paragraph in Significance
    that synthesizes 4-5 prior NIMH-funded digital mental health studies to
    establish the gap. Do not exceed 2,000 words.
    """

    if HUMAN_APPROVED:
        print("\n✓ Human review: APPROVED. Proceeding to final proposal.")
    else:
        print("\n⚠ Human review: PENDING. Edit HUMAN_FEEDBACK and run Cell 10.")

RUN MODE: HUMAN — manual review checkpoint active

Review the draft below. When satisfied:
  1. Set HUMAN_APPROVED = True
  2. Or add feedback to HUMAN_FEEDBACK and run Cell 10

------------------------------------------------------------
**SIGNIFICANCE**

The ongoing mental health crisis in the United States represents a critical public health issue with profound societal ramifications. Approximately 20% of adults and 30% of adolescents experience symptoms indicative of mental health disorders annually, warranting urgent attention (Kessler et al., 2005). Anxiety disorders are particularly prevalent among these conditions, contributing significantly to the overall health burden. However, the treatment uptake remains alarmingly low, with only about half of the affected individuals receiving suitable interventions (Wang et al., 2005). The mental health sector faces a stark treatment gap exacerbated by various barriers, including the stigma associated with mental health disorders, geograp

---

## **Human-Guided Revision**

This cell is only active in `"human"` mode and only executes when `HUMAN_APPROVED = False` and `HUMAN_FEEDBACK` contains substantive editorial guidance. It passes the current draft and the reviewer's feedback to the LLM with an instruction to address all noted deficiencies while preserving the overall structure and maintaining a minimum of 1,700 words. The revised draft is printed in full for review — this cell can be re-run as many times as needed with updated feedback until the draft is satisfactory, at which point `HUMAN_APPROVED` is set to `True` in Cell 9 and the pipeline proceeds to final output.

In [25]:
# ── Human-Guided Revision ─────────────────────────────────────────────────────
if RUN_MODE == "auto":
    print("RUN MODE: AUTO — human revision cell skipped.")
    print("Switch to RUN_MODE = 'human' in Cell 0 to enable this step.")

elif RUN_MODE == "human" and not HUMAN_APPROVED and HUMAN_FEEDBACK.strip():
    print("Incorporating human feedback ...\n")

    human_revision_prompt = f"""You are revising an NIH R01 Technical Approach \
based on feedback from the Principal Investigator. The current draft is \
{len(final_llm_draft.split())} words. Your revised version MUST be at least \
1,700 words — do not shorten any section.

CURRENT DRAFT:
{final_llm_draft}

PI FEEDBACK:
{HUMAN_FEEDBACK}

TASK: Revise the proposal to address all PI feedback while preserving overall \
length and structure. Write in formal NIH proposal prose throughout. \
Do not stop until the full revised proposal is complete."""

    final_llm_draft = llm_complete(
        messages=[{"role": "user", "content": human_revision_prompt}],
        system=PROPOSAL_SYSTEM,
        max_tokens=4000
    )
    revised_wc = len(final_llm_draft.split())
    print(f"Revised draft word count: ~{revised_wc:,} words")
    print("\n=== REVISED DRAFT (post human feedback) ===")
    print(final_llm_draft)
    print("\n→ Review the revised draft above.")
    print("  Set HUMAN_APPROVED = True when ready to proceed to final output.")

elif RUN_MODE == "human" and HUMAN_APPROVED:
    print("✓ Already approved — no revision needed.")

else:
    print("⚠ Add feedback to HUMAN_FEEDBACK and re-run.")

Incorporating human feedback ...

Revised draft word count: ~2,095 words

=== REVISED DRAFT (post human feedback) ===
**SIGNIFICANCE**

The ongoing mental health crisis in the United States constitutes a pivotal public health issue with substantial societal repercussions. Current data reveals that approximately 19.1% of adults experience symptoms associated with mental health disorders annually, necessitating immediate intervention (National Institute of Mental Health, 2021). For adolescents, the figure is even higher, at close to 31.9%, highlighting an urgent need for effective intervention strategies (Merikangas et al., 2010). Anxiety disorders, among other mental health conditions, significantly contribute to the overall health burden, with far-reaching implications for both individuals' well-being and economic productivity (Kessler et al., 2009). Despite the severity, treatment engagement remains insufficient, as evidenced by only 41.4% of adults with any mental illness receiving m

---

## **Final Proposal Output**

This cell prints the final approved draft in full and saves it to a `.txt` file whose name reflects the current run mode — `technical_approach_auto.txt` for autonomous runs and `technical_approach_human.txt` for human-reviewed runs. These two files serve as the primary comparison artifacts for the assignment, demonstrating what the pipeline produces without human intervention versus what it produces after expert editorial review. The `.txt` file also serves as the input to the PDF generation cell that follows.

In [26]:
# ── Final formatting & output ─────────────────────────────────────────────────
print("="*70)
print("FINAL TECHNICAL APPROACH")
print("PAR-25-136 · Laboratories to Optimize Digital Health (R01)")
print(f"Run mode: {RUN_MODE.upper()}")
print("="*70)
print()
print(final_llm_draft)

# ── Save the final draft ──────────────────────────────────────────────────────
if RUN_MODE == "auto":
    output_path = "technical_approach_auto.txt"
    print(f"\n✓ Auto draft saved to: {output_path}")
    print("  Re-run in RUN_MODE = 'human' to produce the human-reviewed version.")
else:
    output_path = "technical_approach_human.txt"
    print(f"\n✓ Human-reviewed draft saved to: {output_path}")

with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"TECHNICAL APPROACH — {RUN_MODE.upper()} RUN\n")
    f.write("PAR-25-136 — Laboratories to Optimize Digital Health (R01)\n")
    f.write("="*70 + "\n\n")
    f.write(final_llm_draft)

FINAL TECHNICAL APPROACH
PAR-25-136 · Laboratories to Optimize Digital Health (R01)
Run mode: HUMAN

**SIGNIFICANCE**

The ongoing mental health crisis in the United States constitutes a pivotal public health issue with substantial societal repercussions. Current data reveals that approximately 19.1% of adults experience symptoms associated with mental health disorders annually, necessitating immediate intervention (National Institute of Mental Health, 2021). For adolescents, the figure is even higher, at close to 31.9%, highlighting an urgent need for effective intervention strategies (Merikangas et al., 2010). Anxiety disorders, among other mental health conditions, significantly contribute to the overall health burden, with far-reaching implications for both individuals' well-being and economic productivity (Kessler et al., 2009). Despite the severity, treatment engagement remains insufficient, as evidenced by only 41.4% of adults with any mental illness receiving mental health serv

---

## **Pipeline Summary & Audit Trail**

This cell prints a concise end-to-end summary of the pipeline run, serving as an audit trail for the assignment submission. It reports the NOFO parsed, the number of chunks indexed, the relevance score and filter decision for every paper in the repository, the number of LLM evaluation rounds completed and their verdicts, whether human approval was granted, and the final draft word count and output file name. Running this cell after both the auto and human passes provides a side-by-side record of how each run progressed through the pipeline.

In [27]:
print("=== PIPELINE SUMMARY ===\n")
print(f"NOFO parsed          : {NOFO_PATH.name}")
print(f"  Chunks indexed     : {len(nofo_chunks)}")
print()
print(f"Papers processed     : {len(papers)}")
for name, score in relevance_scores.items():
    tag = "✓ used" if score >= RELEVANCE_THRESHOLD else "✗ filtered"
    print(f"  {name[:55]:<55} score={score:.3f} [{tag}]")
print()
print(f"Research ideas generated : 5")
print(f"LLM evaluation rounds    : {len(all_evaluations)}")
for ev in all_evaluations:
    print(f"  Round {ev['round']}: {ev['verdict']}")
print()
print(f"Human approved           : {HUMAN_APPROVED}")
print()
print(f"Final draft word count   : ~{len(final_llm_draft.split()):,} words")
print(f"Output saved to          : {output_path}")
print()
print("=== END OF PIPELINE ===")

=== PIPELINE SUMMARY ===

NOFO parsed          : PAR 25 136 Laboratories to Optimize Digital Health.pdf
  Chunks indexed     : 32

Papers processed     : 110
  2021_EPJ_MVMCInfoOps.pdf                                score=0.462 [✓ used]
  2024_ICWSM_Data_Challenge__Post_API_Data_Collection.pdf score=0.434 [✗ filtered]
  23-US-DHS-001.pdf                                       score=0.460 [✓ used]
  AAAI IAA CV.pdf                                         score=0.414 [✗ filtered]
  Acquiring Maintainable AI_Enable Systems_Final.pdf      score=0.378 [✗ filtered]
  ALL18.pdf                                               score=0.353 [✗ filtered]
  Analysis_of_Malware_Communities_Using_Multi_Modal_Featu score=0.388 [✗ filtered]
  Arrow White Paper DExTra.pdf                            score=0.415 [✗ filtered]
  A_Complex_Network_Approach_to_Find_Latent_Terorrist_Com score=0.424 [✗ filtered]
  Benson_MA491_NLP.pdf                                    score=0.454 [✓ used]
  BotBuster___AAAI.pdf  

---

## **PDF Generation**

This cell converts the final approved draft into a formatted PDF using `reportlab`, a pure Python library with no external system dependencies that works reliably on Windows, Mac, and Linux. The `.txt` file is always written first as a fallback in case PDF generation fails. The PDF is formatted to NIH proposal standards — Times New Roman 11pt, justified body text, 1-inch side margins, and 0.75-inch top and bottom margins — with section headers, body paragraphs, and indented Specific Aims rendered in distinct styles. An estimated page count is printed after generation to confirm the draft falls within the 5-page submission limit; if it exceeds 5 pages, section 3c is the recommended place to recover space. Both the `.txt` and `.pdf` outputs are named to reflect the current `RUN_MODE`, producing either `technical_approach_auto.*` or `technical_approach_human.*` for direct comparison between the two runs.

In [28]:
# ── Install reportlab if not already available ────────────────────────────────
# Uncomment and run once if needed:
# import subprocess
# subprocess.run(["pip", "install", "reportlab"], check=True)

# ── PDF Generation (reportlab — no system dependencies required) ──────────────
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib import colors
import re

def generate_pdf(draft: str, run_mode: str) -> None:
    """
    Converts the final proposal draft to a formatted PDF using reportlab.
    Always saves .txt first as a fallback, then attempts PDF generation.
    """
    # ── Always save .txt first ────────────────────────────────────────────────
    txt_path = f"technical_approach_{run_mode}.txt"
    pdf_path = f"technical_approach_{run_mode}.pdf"

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"TECHNICAL APPROACH — {run_mode.upper()} RUN\n")
        f.write("PAR-25-136 — Laboratories to Optimize Digital Health (R01)\n")
        f.write("=" * 70 + "\n\n")
        f.write(draft)
    print(f"✓ Text file saved  : {txt_path}")

    # ── Define styles ─────────────────────────────────────────────────────────
    styles = getSampleStyleSheet()

    style_title = ParagraphStyle(
        name="ProposalTitle",
        fontName="Times-Bold",
        fontSize=13,
        leading=16,
        alignment=TA_CENTER,
        spaceAfter=4,
    )
    style_subtitle = ParagraphStyle(
        name="ProposalSubtitle",
        fontName="Times-Roman",
        fontSize=11,
        leading=14,
        alignment=TA_CENTER,
        spaceAfter=12,
    )
    style_section = ParagraphStyle(
        name="SectionHeader",
        fontName="Times-Bold",
        fontSize=11,
        leading=14,
        spaceBefore=10,
        spaceAfter=4,
    )
    style_body = ParagraphStyle(
        name="BodyText",
        fontName="Times-Roman",
        fontSize=11,
        leading=15,
        alignment=TA_JUSTIFY,
        spaceAfter=6,
    )
    style_aim = ParagraphStyle(
        name="AimText",
        fontName="Times-Roman",
        fontSize=11,
        leading=15,
        alignment=TA_JUSTIFY,
        leftIndent=18,
        spaceAfter=4,
    )

    # ── Build document ────────────────────────────────────────────────────────
    doc = SimpleDocTemplate(
        pdf_path,
        pagesize=letter,
        leftMargin=1.0  * inch,
        rightMargin=1.0 * inch,
        topMargin=0.75 * inch,
        bottomMargin=0.75 * inch,
    )

    story = []

    # Title block
    story.append(Paragraph("TECHNICAL APPROACH", style_title))
    story.append(Paragraph(
        "PAR-25-136 &mdash; Laboratories to Optimize Digital Health (R01)",
        style_subtitle
    ))
    story.append(Spacer(1, 0.05 * inch))

    # ── Parse and render draft content ────────────────────────────────────────
    def format_inline(text: str) -> str:
        """Convert **bold** markers to reportlab <b> tags."""
        return re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', text)

    # Section header patterns to detect and style differently
    section_patterns = [
        r'^#+\s+',              # markdown headers
        r'^\*\*[A-Z0-9]',      # **SECTION** or **3a.
        r'^[0-9]+\.\s+[A-Z]',  # 1. SIGNIFICANCE
        r'^3[a-h]\.',           # 3a. 3b. etc.
    ]
    section_re = re.compile(
        '|'.join(f'({p})' for p in section_patterns)
    )

    paragraphs = draft.split("\n\n")

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue

        # Clean markdown heading markers
        para = re.sub(r'^#+\s+', '', para)

        # Detect section headers
        is_header = bool(section_re.match(para))

        # Handle multi-line paragraphs (single newlines within a block)
        lines = para.split("\n")

        if is_header and len(lines) == 1:
            story.append(Paragraph(format_inline(para), style_section))
        else:
            # Join lines into flowing paragraph
            full_text = " ".join(line.strip() for line in lines if line.strip())
            # Use aim style for indented aim text
            if re.match(r'^(Hypothesis|Primary Analysis|Analysis):', full_text):
                story.append(Paragraph(format_inline(full_text), style_aim))
            else:
                story.append(Paragraph(format_inline(full_text), style_body))

    # ── Build PDF ─────────────────────────────────────────────────────────────
    try:
        doc.build(story)
        print(f"✓ PDF saved        : {pdf_path}")

        # Page count estimate
        word_count = len(draft.split())
        est_pages  = round(word_count / 480, 1)  # ~480 words/page at 11pt TNR
        print(f"  Word count       : ~{word_count:,} words")
        print(f"  Estimated pages  : ~{est_pages} pages")
        if est_pages > 5:
            print(f"  ⚠ May exceed 5-page limit — consider tightening section 3c")
        else:
            print(f"  ✓ Within 5-page limit")

    except Exception as e:
        print(f"⚠ PDF generation failed: {e}")
        print(f"  Text fallback available at: {txt_path}")

# ── Run ───────────────────────────────────────────────────────────────────────
generate_pdf(final_llm_draft, RUN_MODE)

✓ Text file saved  : technical_approach_human.txt
✓ PDF saved        : technical_approach_human.pdf
  Word count       : ~2,095 words
  Estimated pages  : ~4.4 pages
  ✓ Within 5-page limit
